# OpenMAIA / MAIA Demo: Real Neuron Interpretation

#### Colab-ready version of `maia/demo.ipynb` for a real CLIP-RN50 neuron. ####

This notebook keeps the same structure as the original MAIA demo: setup env vars, imports/params, prompt + exemplar loading, tool construction, agent loop, and final inspection.

Default target: `clip-RN50`, `layer4`, unit `1673`.

In [ ]:
%load_ext autoreload
%autoreload 2

## Colab Setup

Run this notebook in a Colab Pro / Pro+ high-RAM A100 runtime. These cells install the real-neuron dependencies, clone MAIA, and apply small runtime patches needed for Colab.

In [ ]:
import os, sys, subprocess
from pathlib import Path

print(sys.version)

def pip_install(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *args])

pip_install(["pip"])

try:
    import torch
    from packaging.version import parse
    torch_ok = parse(torch.__version__.split("+")[0]) >= parse("2.4.0") and torch.cuda.is_available()
    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu", torch.cuda.get_device_name(0))
except Exception:
    torch_ok = False

if not torch_ok:
    pip_install([
        "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
        "--index-url", "https://download.pytorch.org/whl/cu121",
    ])

# Keep binary packages ABI-compatible on Colab Python 3.12.
# If Colab reports a NumPy/Pillow ABI error later, restart the runtime once and rerun from the clone cell.
pip_install([
    "--force-reinstall", "--no-cache-dir",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "scipy==1.13.1",
    "scikit-image==0.25.2",
    "Pillow==10.4.0",
])

pip_install([
    "diffusers", "transformers", "accelerate", "bitsandbytes", "safetensors",
    "sentencepiece", "protobuf", "hf_transfer", "openai", "anthropic", "timm", "peft",
    "IPython", "tiktoken", "statsmodels", "tabulate", "ftfy", "regex", "tqdm", "einops",
    "matplotlib",
])

pip_install([
    "git+https://github.com/openai/CLIP.git",
    "git+https://github.com/davidbau/baukit@9d51abd51ebf29769aecc38c4cbef459b731a36e",
])

print("Dependency install complete. If this was the first install in a fresh runtime, restart once, then continue.")


In [ ]:
from pathlib import Path

MAIA_DIR = Path("/content/maia")
if not (MAIA_DIR / ".git").exists():
    !git clone https://github.com/AtharvRN/maia.git /content/maia
else:
    print("Using existing /content/maia")
    !cd /content/maia && git pull --ff-only

!cd /content/maia && python -m py_compile maia_api.py main.py utils/agents/adapters.py utils/agents/factory.py utils/agents/messages.py utils/flux.py utils/flux_kontext.py


### Setup Env. Vars

In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

# Set your key here, or use Colab Secrets below.
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "<your_openai_api_key>")
os.environ["OPENAI_ORGANIZATION"] = os.environ.get("OPENAI_ORGANIZATION", "")

# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ["OPENAI_API_KEY"] and "your_openai" not in os.environ["OPENAI_API_KEY"], "Set OPENAI_API_KEY first."

### Imports and Params

In [ ]:
%cd /content/maia

import random
import json
import torch
import os

from maia_api import Synthetic_System, System, Tools
from utils.agents.factory import create_agent
from utils.DatasetExemplars import DatasetExemplars
from utils.ExperimentEnvironment import ExperimentEnvironment
from utils.flux import FluxDev
from utils.flux_kontext import FluxKontextDev
from utils.main_utils import *
from utils.SyntheticExemplars import SyntheticExemplars

random.seed(0000)

In [ ]:
# Layers to explore for each model
layers = {
    'resnet152': ['conv1', 'layer1', 'layer2', 'layer3', 'layer4'],
    'clip-RN50': ['layer1', 'layer2', 'layer3', 'layer4'],
    'dino_vits8': [
        'blocks.1.mlp.fc1',
        'blocks.3.mlp.fc1',
        'blocks.5.mlp.fc1',
        'blocks.7.mlp.fc1',
        'blocks.9.mlp.fc1',
        'blocks.11.mlp.fc1',
    ],
    'synthetic_neurons': ['mono', 'or', 'and'],
}

In [ ]:
agent_name: str = 'gpt-4o'  # use 'gpt-4o-mini' if your TPM limit is tight
base_url: str = 'http://localhost:11434/v1'  # only for local agents
task: str = 'neuron_description'

model: str = 'clip-RN50'
layer: str = 'layer4'
unit: int = 1673
mode: str = "manual"

path2save: str = '/content/maia_colab_results'
path2prompts: str = './prompts/open/'
path2exemplars: str = './exemplars'

device: str = "0"
max_output_tokens: int = 1024
max_rounds: int = 15

# Single A100: both local image tools target cuda:0 with CPU offload enabled.
text2image_device: str = 'cuda:0'
img2img_device: str = 'cuda:0'


### Optional Memory Monitor

This writes CPU/GPU usage to `/content/maia/logs/colab_memory_monitor.log`.

In [ ]:
import subprocess, os

os.makedirs('/content/maia/logs', exist_ok=True)
monitor_path = '/content/maia/logs/colab_memory_monitor.log'
monitor_cmd = """
while true; do
  echo "===== $(date -Iseconds) ====="
  echo "--- CPU RAM ---"
  free -h
  echo "--- GPU ---"
  nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu,power.draw --format=csv,noheader,nounits || true
  echo "--- GPU processes ---"
  nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits || true
  echo "--- Top RSS ---"
  ps -eo pid,etime,%cpu,%mem,rss,cmd --sort=-rss | head -15
  sleep 10
done
"""
monitor_proc = subprocess.Popen(['bash', '-lc', monitor_cmd], stdout=open(monitor_path, 'w'), stderr=subprocess.STDOUT)
print('memory monitor pid:', monitor_proc.pid)
print('memory log:', monitor_path)

### Reading Prompts, Loading NetDissect Exemplars, Building Tools and Environment

In [ ]:
# Read the API and User prompt
maia_api, user_query = return_prompt(path2prompts, setting=task)

# Load NetDissect Exemplars
unit = int(unit)
if model == 'synthetic_neurons':
    net_dissect = SyntheticExemplars(os.path.join(path2exemplars, model), path2save, layer)
    with open(os.path.join('./synthetic_neurons_dataset/labels/', f'{layer}.json')) as f:
        synthetic_neuron_data = json.load(f)
else:
    net_dissect = DatasetExemplars(path2exemplars, path2save, model, layer, [unit])

# Create directory to save results
path2save = os.path.join(path2save, agent_name, model, str(layer), str(unit))
os.makedirs(path2save, exist_ok=True)

# Setup the system to explore
if model == 'synthetic_neurons':
    gt_label = synthetic_neuron_data[unit]['label'].rsplit('_')
    print('groundtruth label:', gt_label)
    system = Synthetic_System(unit, gt_label, layer, device)
else:
    system = System(unit, layer, model, device, net_dissect.thresholds)

# Initialize tools and experiment environment
print(f'Loading FluxDev on {text2image_device}')
text2image_model = FluxDev(device=text2image_device)
print(f'Loading FluxKontextDev on {img2img_device}')
img2img_model = FluxKontextDev(device=img2img_device)
tools = Tools(
    path2save,
    device,
    net_dissect,
    image2text_model_name=agent_name,
    text2image_model=text2image_model,
    img2img_model=img2img_model,
)
experiment_env = ExperimentEnvironment(system, tools, globals())


### Initialize the Agent and Start the Experimentation Loop

In [ ]:
# Start the experiment log with the system prompt (maia api) and the user prompt (the query)
tools.update_experiment_log(role='system', type='text', type_content=maia_api)
tools.update_experiment_log(role='user', type='text', type_content=user_query)
ind = len(tools.experiment_log)

# Create the Agent 
agent = create_agent(
    model=agent_name,
    max_attempts=5,
    max_output_tokens=max_output_tokens,
    **({'base_url': base_url} if 'local' in agent_name else {}),
)
round_count = 0
while True:
    round_count += 1
    # Ask MAIA to provide the next experiment to execute
    maia_experiment = agent.ask(tools.experiment_log)
    if maia_experiment is None:
        tools.update_experiment_log(
            role='user',
            type='text',
            type_content='Agent returned None after retries; stopping this unit.',
        )
        tools.generate_html(path2save)
        break

    # Log MAIA's response
    tools.update_experiment_log(role='maia', type='text', type_content=str(maia_experiment))
    plot_results_notebook(tools.experiment_log[ind:])
    ind = len(tools.experiment_log)
    tools.generate_html(path2save)  # HTML log after each MAIA response
    
    # If we exceed 15 rounds, we force MAIA to finish
    if round_count > max_rounds:  
        overload_instructions(tools, prompt_path=path2prompts)
    # Check for stopping condition
    if '[DESCRIPTION]' in maia_experiment:
        break
    try:
        # Execute the experiment suggested by MAIA
        output = experiment_env.execute_experiment(maia_experiment)
        # Log the experiment output
        if output:
            tools.update_experiment_log(role='user', type='text', type_content=output)
    except Exception as exec_e:
        tools.update_experiment_log(role='user', type='text',
            type_content=f'Error during experiment execution: {str(exec_e)}',
        )

# Save the final dialogue log as a JSON file
save_dialogue(tools.experiment_log, path2save)

### Inspect Final Result

In [ ]:
from pathlib import Path

result_path = Path(path2save)
print('HTML log:', result_path / 'experiment.html')
print('History:', result_path / 'history.json')

for msg in reversed(tools.experiment_log):
    content = msg.get('content', '')
    if isinstance(content, str) and '[DESCRIPTION]' in content:
        print(content)
        break

### MAIA API

In [ ]:
print(maia_api)

### Interpretability task

In [ ]:
print(user_query)